<a href="https://colab.research.google.com/github/tarasovEgor/DeepLearningKurs/blob/main/src/lab_2/llm_transformer_lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 2.**

---

## LLM и Трансформер

---

## **Цель работы:** подготовить компактный корпус немецкоязычных диалогов, изучить ключевые компоненты архитектуры трансформера, выполнить fine-tuning GPT-подобной модели для генерации диалоговых ответов, сравнить оптимизаторы и scheduler, оценить качество по метрикам Perplexity, BLEU, ROUGE и протестировать модель в интерактивном режиме.


### 0. Init steps

---

#### 0.1 Установка дополнительных модулей:

In [1]:
!pip -q install transformers datasets evaluate rouge-score sentencepiece accelerate nltk spacy scikit-learn matplotlib pandas
!python -m spacy download de_core_news_sm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 43.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#### 0.2 Импорты библиотек и базовая настройка окружения:

In [6]:
import os
import re
import math
import json
import random
import warnings
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import spacy
from rouge_score import rouge_scorer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

Устройство: cuda


#### 0.3 Загрузка ресурсов NLTK и spaCy:

In [3]:
nltk.download("punkt")
nltk.download("stopwords")

nlp = spacy.load("de_core_news_sm", disable=["parser", "ner"])

german_stopwords = set(stopwords.words("german"))

print("Количество немецких стоп-слов:", len(german_stopwords))
print("spaCy модель загружена успешно")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Количество немецких стоп-слов: 232
spaCy модель загружена успешно


#### 0.4 Общая конфигурация проекта

In [12]:
MODEL_NAME = "dbmdz/german-gpt2"

DATA_URL = "https://raw.githubusercontent.com/liuzeming01/XDailyDialog/main/data/1k_part_data/dialogues_text_De.txt"
SAMPLE_SIZE = None

# Базовые параметры
MAX_LENGTH = 96
BATCH_SIZE = 4
NUM_EPOCHS = 2

LR_ADAM = 5e-5
LR_SGD = 1e-3
LR_RMSPROP = 1e-4

MODEL_SAVE_DIR = "/content/german_dialog_gpt2"
LOG_FILE = "/content/chat_log.txt"

print("Модель:", MODEL_NAME)
print("Источник датасета:", DATA_URL)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)

Модель: dbmdz/german-gpt2
Источник датасета: https://raw.githubusercontent.com/liuzeming01/XDailyDialog/main/data/1k_part_data/dialogues_text_De.txt
SAMPLE_SIZE: None
MAX_LENGTH: 96
BATCH_SIZE: 4
NUM_EPOCHS: 2


#### 0.5 Проверка токенизатора:

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

test_text = "Hallo! Wie geht es dir heute?"
tokens = tokenizer.tokenize(test_text)
token_ids = tokenizer.encode(test_text)

print("Тестовая строка:", test_text)
print("Токены:", tokens)
print("ID токенов:", token_ids[:20])
print("PAD token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/865 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Тестовая строка: Hallo! Wie geht es dir heute?
Токены: ['Hallo', '!', 'ĠWie', 'Ġgeht', 'Ġes', 'Ġdir', 'Ġheute', '?']
ID токенов: [5568, 5, 2675, 1284, 444, 1112, 1329, 35]
PAD token: <|endoftext|>


#### 0.6 Вывод по нулевому блоку:

На начальном этапе были установлены и подключены библиотеки для обработки текста, обучения трансформерной модели и расчёта метрик качества. В качестве основы для корпуса выбран немецкий файл dialogues_text_De.txt из датасета XDailyDialog, содержащего многоходовые диалоги. Для fine-tuning выбрана предобученная модель dbmdz/german-gpt2, ориентированная на немецкий язык. Дополнительно были загружены ресурсы nltk и spaCy для токенизации, удаления стоп-слов и лемматизации текста.

### 1. Загрузка данных

---

#### 1.1 Загрузка немецкого корпуса диалогов XDailyDialog:

Каждая строка - один диалог, а реплики внутри строки разделены специальным маркером __eou__:

In [13]:

SAMPLE_SIZE = None

data_path = "/content/dialogues_text_De.txt"
urllib.request.urlretrieve(DATA_URL, data_path)

print("Файл сохранён:", data_path)
print("Размер файла (байт):", os.path.getsize(data_path))

Файл сохранён: /content/dialogues_text_De.txt
Размер файла (байт): 479639


#### 1.2 Чтение корпуса и первичный просмотр:

In [15]:
with open(data_path, "r", encoding="utf-8") as f:
    raw_dialogues = [line.strip() for line in f.readlines() if line.strip()]

print("Количество диалогов в корпусе:", len(raw_dialogues))
print("\nПример первого диалога:\n")
print(raw_dialogues[0])

print("\n" + "-"*80 + "\n")
print("Пример второго диалога:\n")
print(raw_dialogues[1])

Количество диалогов в корпусе: 1000

Пример первого диалога:

Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __eou__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __eou__ Was hätte ich mitbringen sollen? __eou__ Das einzige, was ich sehen muss, ist Ihr Führerschein, da ich diese Papiere beglaubigen werde. __eou__ Ich fühle mich ein wenig überwältigt von so vielen Papieren. __eou__ Machen Sie sich keine Sorgen, wie viele Papiere es sind. __eou__ Mein Freund ist Anwalt und sagte mir, dass ich ihm alles faxen kann, wenn ich eine Frage habe. __eou__ Bitte holen Sie sich Hilfe von außen, die Sie für das Verständnis Ihrer Escrow-Dokumente benötigen. __eou__ Ist das das Letzte, was ich tun muss, bevor das Haus mir gehört? __eou__ Am Ende des Schecks wird das Haus Ihnen gehören!

--------------------------------------------------------------------------------

Пример второго диалога:

Smith ist immer unvorsichtig,

#### 1.3 Разбор диалогов на отдельные реплики:

В XDailyDialog реплики внутри строки разделены токеном __eou__.
Мы разбиваем каждый диалог на список реплик и удаляем пустые элементы.
Также оставляем только те диалоги, где есть минимум 2 реплики, иначе невозможно построить пару вопрос–ответ. Формат с __eou__ используется и в DailyDialog-подобных представлениях корпуса.

In [17]:
def split_dialogue(dialogue_text):
    utterances = [utt.strip() for utt in dialogue_text.split("eou") if utt.strip()]
    return utterances

parsed_dialogues = [split_dialogue(dialogue) for dialogue in raw_dialogues]
parsed_dialogues = [dialogue for dialogue in parsed_dialogues if len(dialogue) >= 2]

print("Количество пригодных диалогов:", len(parsed_dialogues))
print("\nПример разобранного диалога:")
print(parsed_dialogues[0])
print("\nКоличество реплик в первом диалоге:", len(parsed_dialogues[0]))

dialogue_lengths = [len(dialogue) for dialogue in parsed_dialogues]

print("Минимум реплик в диалоге:", min(dialogue_lengths))
print("Максимум реплик в диалоге:", max(dialogue_lengths))
print("Среднее число реплик в диалоге:", round(np.mean(dialogue_lengths), 2))
print("Медиана:", np.median(dialogue_lengths))

Количество пригодных диалогов: 1000

Пример разобранного диалога:
['Ich bin sehr nervös, meine Treuhandpapiere zu unterschreiben. __', '__ Escrow beinhaltet eine Menge Papierkram, aber ich werde Ihnen alle Schritte erklären, während wir weitermachen. __', '__ Was hätte ich mitbringen sollen? __', '__ Das einzige, was ich sehen muss, ist Ihr Führerschein, da ich diese Papiere beglaubigen werde. __', '__ Ich fühle mich ein wenig überwältigt von so vielen Papieren. __', '__ Machen Sie sich keine Sorgen, wie viele Papiere es sind. __', '__ Mein Freund ist Anwalt und sagte mir, dass ich ihm alles faxen kann, wenn ich eine Frage habe. __', '__ Bitte holen Sie sich Hilfe von außen, die Sie für das Verständnis Ihrer Escrow-Dokumente benötigen. __', '__ Ist das das Letzte, was ich tun muss, bevor das Haus mir gehört? __', '__ Am Ende des Schecks wird das Haus Ihnen gehören!']

Количество реплик в первом диалоге: 10
Минимум реплик в диалоге: 2
Максимум реплик в диалоге: 30
Среднее число реплик в